# 05 · Perturbation effect estimation (pseudobulk delta matrix)
For each perturbation *p*: **delta_p = mean log-expression(cells with p) − mean log-expression(control cells)**, computed batch-matched when a batch column exists. The matrix *perturbations × genes* is the target the ML models learn to predict.

We also estimate a **noise ceiling**: split each perturbation's cells in two halves, estimate delta from each, and correlate. No model can be expected to beat this reproducibility.

In [ ]:
# ---- Bootstrap: Colab (Google Drive) or local ----
import os, sys, subprocess
REPO_URL = "https://github.com/<your-username>/PerturbAI.git"   # <-- edit once after you create the repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/PerturbAI"):
        subprocess.run(["git", "clone", REPO_URL, "/content/PerturbAI"], check=True)
    os.chdir("/content/PerturbAI")
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
else:
    try:
        import perturbai  # installed with `pip install -e .`
    except ImportError:
        sys.path.insert(0, os.path.abspath("../src"))  # fallback when running from notebooks/
from perturbai.paths import get_paths
P = get_paths()
print("Data root:", P.root)

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, matplotlib.pyplot as plt
from perturbai.paths import artifact
from perturbai.utils import set_seed, load_config
CFG = load_config("data_config.yaml"); MCFG = load_config("model_config.yaml")
set_seed(CFG["seed"])
SYNTHETIC = False        # True = 1-minute smoke test on simulated data (no download)
DS = "replogle_k562_essential"
INFO = CFG["datasets"][DS]
PERT, CTRL, BATCH = INFO["perturbation_col"], INFO["control_label"], INFO["batch_col"]
TAG = DS + ("_synthetic" if SYNTHETIC else "")   # artifact prefix

In [ ]:
import scanpy as sc
from perturbai.features.effects import effects_from_adata, split_half_effects, effect_size_summary
adata = sc.read_h5ad(P.processed / artifact("subset_adata", TAG))
batch_col = BATCH if BATCH in adata.obs else None
res = effects_from_adata(adata, PERT, CTRL, batch_col, min_cells=20 if not SYNTHETIC else 10)
res.delta.to_parquet(P.processed / artifact("effects", TAG))
res.ctrl_mean.to_frame("ctrl_mean").to_parquet(P.processed / artifact("ctrl_mean", TAG))
res.n_cells.rename("n_cells").to_csv(P.processed / artifact("n_cells", TAG))
print(res.delta.shape, "| median cells/perturbation:", res.n_cells.median())

In [ ]:
S = effect_size_summary(res.delta).join(res.n_cells.rename("n_cells"))
kd = pd.read_csv(P.interim / f"{TAG}_knockdown.csv", index_col=0)
S = S.join(kd[["knockdown"]])
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
ax[0].hist(S.l2_norm, bins=40); ax[0].set(title="Effect size (L2 of delta)")
ax[1].scatter(S.n_cells, S.l2_norm, s=8); ax[1].set(xlabel="cells", ylabel="L2", title="Effect vs cells")
ax[2].scatter(S.knockdown, S.l2_norm, s=8); ax[2].set(xlabel="knockdown", ylabel="L2", title="Effect vs knockdown")
plt.tight_layout(); plt.show()
S.sort_values("l2_norm", ascending=False).head(10)

In [ ]:
from perturbai.evaluation.metrics import rowwise_pearson, rowwise_spearman
labels = adata.obs[PERT].astype(str).values
a, b = split_half_effects(adata.X, labels, adata.var_names, CTRL,
                          None if batch_col is None else adata.obs[batch_col].values,
                          min_cells=20 if not SYNTHETIC else 10, seed=0)
ceil = pd.DataFrame({"pearson_split_half": rowwise_pearson(a.values, b.values),
                     "spearman_split_half": rowwise_spearman(a.values, b.values)}, index=a.index)
ceil.to_csv(P.results / f"{TAG}_noise_ceiling.csv")
print(ceil.describe().round(3))
plt.hist(ceil.pearson_split_half.dropna(), bins=30); plt.xlabel("split-half Pearson (delta)"); plt.title("Noise ceiling"); plt.show()

**Interpretation:** many perturbations have weak effects; their deltas are mostly noise (low split-half correlation). Expect the model benchmark to look worse on these, which is a property of the data, not necessarily the model.